# Module 2: Online Retail Analytics — RFM Segmentation & Market Basket

Welcome to **Module 2**! In this lab, you will analyze 541,900 transaction records from a UK giftware wholesaler (`online_retail.xlsx`).

### Business Brief:
> *"Which customers and products actually make us money, and who is about to stop buying?"*

---

### Assignment Tasks Overview:
1. **Task 1**: Load `online_retail.xlsx` using Pandas, clean missing `CustomerID`s, and filter cancelled orders (Invoice starting with 'C').
2. **Task 2**: Calculate Total Sales (`Quantity * UnitPrice`) and filter invalid non-positive quantities.
3. **Task 3**: Monthly Revenue Analysis & Top Performing Countries.
4. **Task 4**: Customer RFM Segmentation (Calculate Recency, Frequency, Monetary scores).
5. **Task 5**: Market Basket Analysis — Discover product association rules for cross-selling recommendations.

In [1]:
# Task 1: Data Sanitization & Cleaning

import pandas as pd
import numpy as np

# Load CSV dataset
df = pd.read_csv('online_retail.csv')

print("Original Dataset Shape:", df.shape)

# Remove rows without Customer ID
df_clean = df.dropna(subset=['CustomerID']).copy()

# Convert InvoiceNo to string
df_clean['InvoiceNo'] = df_clean['InvoiceNo'].astype(str)

# Remove cancelled orders
df_clean = df_clean[
    ~df_clean['InvoiceNo'].str.startswith('C')
].copy()

# Remove invalid quantities and prices
df_clean = df_clean[
    (df_clean['Quantity'] > 0) &
    (df_clean['UnitPrice'] > 0)
].copy()

# Calculate Total Sales
df_clean['TotalSales'] = (
    df_clean['Quantity'] * df_clean['UnitPrice']
)

print("Cleaned Dataset Shape:", df_clean.shape)

print("\nFirst 10 Cleaned Rows:")
print(df_clean.head(10))

Original Dataset Shape: (541909, 8)
Cleaned Dataset Shape: (397884, 9)

First 10 Cleaned Rows:
  InvoiceNo StockCode  ...         Country  TotalSales
0    536365    85123A  ...  United Kingdom       15.30
1    536365     71053  ...  United Kingdom       20.34
2    536365    84406B  ...  United Kingdom       22.00
3    536365    84029G  ...  United Kingdom       20.34
4    536365    84029E  ...  United Kingdom       20.34
5    536365     22752  ...  United Kingdom       15.30
6    536365     21730  ...  United Kingdom       25.50
7    536366     22633  ...  United Kingdom       11.10
8    536366     22632  ...  United Kingdom       11.10
9    536367     84879  ...  United Kingdom       54.08

[10 rows x 9 columns]



### Task 2: RFM Customer Segmentation
Calculate Recency (days since last order), Frequency (order count), and Monetary (total spend) for each customer.

In [2]:
# Task 2: Total Sales Analysis

print("Total Revenue:", df_clean['TotalSales'].sum())

print("Average Transaction Value:", df_clean['TotalSales'].mean())

print("Minimum Transaction Value:", df_clean['TotalSales'].min())

print("Maximum Transaction Value:", df_clean['TotalSales'].max())

print("\nInvalid Quantity Records:",
      (df_clean['Quantity'] <= 0).sum())

print("Invalid UnitPrice Records:",
      (df_clean['UnitPrice'] <= 0).sum())

Total Revenue: 8911407.904
Average Transaction Value: 22.396999889415003
Minimum Transaction Value: 0.001
Maximum Transaction Value: 168469.6

Invalid Quantity Records: 0
Invalid UnitPrice Records: 0



In [3]:
# Task 3: Monthly Revenue Analysis & Top Performing Countries

# Convert InvoiceDate to datetime
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

# Create Month column
df_clean['Month'] = df_clean['InvoiceDate'].dt.to_period('M')

# Monthly Revenue
monthly_revenue = (
    df_clean
    .groupby('Month', as_index=False)['TotalSales']
    .sum()
)

monthly_revenue['TotalSales'] = monthly_revenue['TotalSales'].round(2)

print("=== Monthly Revenue ===")
print(monthly_revenue.to_string(index=False))

# Top Performing Countries
country_revenue = (
    df_clean
    .groupby('Country', as_index=False)['TotalSales']
    .sum()
    .sort_values('TotalSales', ascending=False)
)

country_revenue['TotalSales'] = country_revenue['TotalSales'].round(2)

print("\n=== Top 10 Countries by Revenue ===")
print(country_revenue.head(10).to_string(index=False))

=== Monthly Revenue ===
  Month  TotalSales
2010-12   572713.89
2011-01   569445.04
2011-02   447137.35
2011-03   595500.76
2011-04   469200.36
2011-05   678594.56
2011-06   661213.69
2011-07   600091.01
2011-08   645343.90
2011-09   952838.38
2011-10  1039318.79
2011-11  1161817.38
2011-12   518192.79

=== Top 10 Countries by Revenue ===
       Country  TotalSales
United Kingdom  7308391.55
   Netherlands   285446.34
          EIRE   265545.90
       Germany   228867.14
        France   209024.05
     Australia   138521.31
         Spain    61577.11
   Switzerland    56443.95
       Belgium    41196.34
        Sweden    38378.33



In [4]:
# Task 4: Customer RFM Segmentation

# Set analysis date as one day after the last transaction
analysis_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)

# Calculate RFM metrics
rfm = (
    df_clean
    .groupby('CustomerID')
    .agg(
        Recency=('InvoiceDate', lambda x:
                 (analysis_date - x.max()).days),
        Frequency=('InvoiceNo', 'nunique'),
        Monetary=('TotalSales', 'sum')
    )
    .reset_index()
)

# Create RFM scores
rfm['Recency_Score'] = pd.qcut(
    rfm['Recency'],
    4,
    labels=[4, 3, 2, 1]
)

rfm['Frequency_Score'] = pd.qcut(
    rfm['Frequency'].rank(method='first'),
    4,
    labels=[1, 2, 3, 4]
)

rfm['Monetary_Score'] = pd.qcut(
    rfm['Monetary'].rank(method='first'),
    4,
    labels=[1, 2, 3, 4]
)

# Convert scores to integers
rfm['Recency_Score'] = rfm['Recency_Score'].astype(int)
rfm['Frequency_Score'] = rfm['Frequency_Score'].astype(int)
rfm['Monetary_Score'] = rfm['Monetary_Score'].astype(int)

# Combined RFM Score
rfm['RFM_Score'] = (
    rfm['Recency_Score'].astype(str) +
    rfm['Frequency_Score'].astype(str) +
    rfm['Monetary_Score'].astype(str)
)

print("=== RFM Summary ===")
print(rfm.head(10).to_string(index=False))

print("\nTotal Customers:", len(rfm))

print("\nRFM Score Distribution:")
print(rfm['RFM_Score'].value_counts().head(15))

=== RFM Summary ===
 CustomerID  Recency  Frequency  Monetary  Recency_Score  Frequency_Score  Monetary_Score RFM_Score
    12346.0      326          1  77183.60              1                1               4       114
    12347.0        2          7   4310.00              4                4               4       444
    12348.0       75          4   1797.24              2                3               4       234
    12349.0       19          1   1757.55              3                1               4       314
    12350.0      310          1    334.40              1                1               2       112
    12352.0       36          8   2506.04              3                4               4       344
    12353.0      204          1     89.00              1                1               1       111
    12354.0      232          1   1079.40              1                1               3       113
    12355.0      214          1    459.40              1                1       

In [6]:
# Task 5: Market Basket Analysis without mlxtend

from itertools import combinations
from collections import Counter

# Keep required columns
basket_data = df_clean[
    ['InvoiceNo', 'StockCode']
].copy()

# Remove duplicate product entries within the same invoice
basket_data = basket_data.drop_duplicates()

# Create invoice -> products mapping
invoice_products = (
    basket_data
    .groupby('InvoiceNo')['StockCode']
    .apply(set)
)

# Count individual products
item_counts = Counter()

# Count product pairs
pair_counts = Counter()

for products in invoice_products:
    for item in products:
        item_counts[item] += 1

    for pair in combinations(sorted(products), 2):
        pair_counts[pair] += 1

# Total number of invoices
total_invoices = len(invoice_products)

# Create frequent product pairs
pair_results = []

for pair, count in pair_counts.items():
    support = count / total_invoices

    if support >= 0.02:
        pair_results.append({
            'Product_1': pair[0],
            'Product_2': pair[1],
            'Transactions_Together': count,
            'Support': support
        })

pair_results = pd.DataFrame(pair_results)

if len(pair_results) > 0:

    # Calculate confidence for both directions
    pair_results['Confidence_Product1_to_Product2'] = (
        pair_results['Transactions_Together'] /
        pair_results['Product_1'].map(item_counts)
    )

    pair_results['Confidence_Product2_to_Product1'] = (
        pair_results['Transactions_Together'] /
        pair_results['Product_2'].map(item_counts)
    )

    # Show strongest pairs by support
    pair_results = pair_results.sort_values(
        'Support',
        ascending=False
    )

    print("Total Transactions:", total_invoices)

    print("\n=== Top Product Associations ===")
    print(
        pair_results.head(20).to_string(index=False)
    )

else:
    print("No frequent product pairs found with support >= 2%.")

Total Transactions: 18532

=== Top Product Associations ===
Product_1 Product_2  Transactions_Together  Support  Confidence_Product1_to_Product2  Confidence_Product2_to_Product1
    22386    85099B                    546 0.029463                         0.626866                         0.341250
    22697     22699                    541 0.029193                         0.782923                         0.690932
    22726     22727                    530 0.028599                         0.671736                         0.604333
    20725     22384                    523 0.028221                         0.405741                         0.561762
    20725     22383                    519 0.028006                         0.402638                         0.497603
    20725     20727                    517 0.027898                         0.401086                         0.491445
    82482    82494L                    468 0.025254                         0.534247                         0.577